# 07 - Train XGBoost (eXtreme Gradient Boosting)

Train XGBoost classifier for toxicity prediction.

**Key Features:**
- Gradient boosting with decision trees
- Handles class imbalance with scale_pos_weight
- Feature importance extraction

In [1]:
# ============================================================================
# IMPORTS AND CONFIG
# ============================================================================
import os, sys, pickle
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

PARAM_GRID = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

TOXICITY_ENDPOINTS = ['NR-AhR', 'NR-AR', 'NR-AR-LBD', 'NR-Aromatase',
                      'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma',
                      'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']
MODELS_DIR = '../models/baseline_models'
print("✓ Setup complete")

✓ Setup complete


In [ ]:
# ============================================================================
# TRAINING FUNCTION
# ============================================================================
def train_xgboost(toxicity_name):
    """Train XGBoost for a single toxicity endpoint."""
    print(f"\nTraining XGBoost for {toxicity_name}...")
    
    cache_path = f'../Data/cache/{toxicity_name}/splits.pkl'
    if not os.path.exists(cache_path):
        return None
    
    with open(cache_path, 'rb') as f:
        data = pickle.load(f)
    
    X_train_val = np.vstack([data['train']['X'], data['val']['X']])
    y_train_val = np.concatenate([data['train']['y'], data['val']['y']])
    X_test, y_test = data['test']['X'], data['test']['y']
    
    # Calculate scale_pos_weight for class imbalance
    neg_count = np.sum(y_train_val == 0)
    pos_count = np.sum(y_train_val == 1)
    scale_pos_weight = neg_count / pos_count if pos_count > 0 else 1.0
    
    # Grid search
    xgb = XGBClassifier(random_state=42, eval_metric='logloss',
                        scale_pos_weight=scale_pos_weight, n_jobs=-1)
    grid_search = GridSearchCV(xgb, PARAM_GRID, cv=5, scoring='roc_auc', n_jobs=-1)
    grid_search.fit(X_train_val, y_train_val)
    
    # Evaluate
    best_model = grid_search.best_estimator_
    y_proba = best_model.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, y_proba)
    
    print(f"  n_estimators={best_model.n_estimators}, lr={best_model.learning_rate}, AUC: {test_auc:.4f}")
    
    # Save
    os.makedirs(f'{MODELS_DIR}/{toxicity_name}', exist_ok=True)
    with open(f'{MODELS_DIR}/{toxicity_name}/XGBoost_model.pkl', 'wb') as f:
        pickle.dump({'model': best_model}, f)
    
    return {'test_auc': test_auc}

result = train_xgboost('NR-AhR')